In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/rt-detrv2-fine-tune/RT-DETR/rtdetrv2_pytorch

/content/drive/MyDrive/rt-detrv2-fine-tune/RT-DETR/rtdetrv2_pytorch


In [ ]:
!pip install supervision
!pip install torchmetrics
!pip install albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 19.4 MB/s eta 0:00:00


## 1. Load fine-tune model

In [ ]:
# load model
import torch
from argparse import Namespace
from src.core import YAMLConfig
import torch.nn as nn
import torch.nn.init as init

args = Namespace(config_path='configs/rtdetrv2/rtdetrv2_r18vd_120e_coco.yml',
                 resume_path='models/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth',
                 tuning=None, device=None, seed=0, use_amp=True, output_dir=None,
                 summary_dir=None, test_only=False, update=None, print_method='builtin',
                 print_rank=0, local_rank=None)


def load_pretrained_model(config_path,resume_path):

    # initialize the raw model
    cfg=YAMLConfig(config_path, resume=resume_path)
    model=cfg.model
    # model state_dict
    state_dict_model=model.state_dict()

    # pretrained state_dict
    checkpoint=torch.load(args.resume_path,map_location="cpu")
    if 'ema' in checkpoint:
        state_dict_pretrained=checkpoint['ema']['module']
    else:
        state_dict_pretrained=checkpoint['model']

    # Create a new state dictionary to store matched weights
    matched_weights = {}

        # Loop through all layers in the model
    for model_key, model_param in state_dict_model.items():
        # Try to find a matching key in the state_dict
        matched_key = None
        for state_key in state_dict_pretrained.keys():
            # Check if the state_dict key is a substring of the model key
            if state_key in model_key:
                matched_key = state_key
                break

        # If a matching key is found and shapes match, load the weight
        if matched_key is not None:
            state_weight = state_dict_pretrained[matched_key]

            # Ensure the shapes match exactly
            if state_weight.shape == model_param.shape:
                matched_weights[model_key] = state_weight
                print(f"Matched and loaded weight for: {model_key}")
            else:
                print(f"Shape mismatch for {model_key}: {state_weight.shape} vs {model_param.shape}")

    # Load the matched weights into the model
    model.load_state_dict(matched_weights, strict=False)
    print(f"\nLoad pretrained weights succesfully | {sum(p.numel() for p in model.parameters())/1e6} million parameters")
    return model, cfg.criterion

model, criterion=load_pretrained_model(args.config_path,args.resume_path)


Load PResNet18 state_dict


Matched and loaded weight for: backbone.conv1.conv1_1.conv.weight
Matched and loaded weight for: backbone.conv1.conv1_1.norm.weight
Matched and loaded weight for: backbone.conv1.conv1_1.norm.bias
Matched and loaded weight for: backbone.conv1.conv1_1.norm.running_mean
Matched and loaded weight for: backbone.conv1.conv1_1.norm.running_var
Matched and loaded weight for: backbone.conv1.conv1_1.norm.num_batches_tracked
Matched and loaded weight for: backbone.conv1.conv1_2.conv.weight
Matched and loaded weight for: backbone.conv1.conv1_2.norm.weight
Matched and loaded weight for: backbone.conv1.conv1_2.norm.bias
Matched and loaded weight for: backbone.conv1.conv1_2.norm.running_mean
Matched and loaded weight for: backbone.conv1.conv1_2.norm.running_var
Matched and loaded weight for: backbone.conv1.conv1_2.norm.num_batches_tracked
Matched and loaded weight for: backbone.conv1.conv1_3.conv.weight
Matched and loaded weight for: backbone.conv1.conv1_3.norm.weight
Matched and loaded weight for: b

## 2. Data

In [ ]:
import albumentations as A
from data_visdrone import VisDroneData
from torch.utils.data import DataLoader

def collate_fn(batch):
    # Extract pixel values and labels
    pixel_values = torch.stack([x["pixel_values"] for x in batch])

    # Prepare labels
    labels = [x["labels"] for x in batch]

    return {"pixel_values": pixel_values, "labels": labels}

# dataloaders
train_transform = A.Compose(
    [
        A.Perspective(p=0.1),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(p=0.1),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",  # Albumentations expects [xmin, ymin, xmax, ymax]
        label_fields=["category"],
        clip=True,
        min_area=1,
    ),
)

val_transform = A.Compose(
    [A.NoOp()],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["category"],
        clip=True,
        min_area=1,
    ),
)

ds_train = VisDroneData(
        json_path="dataset/visdrone/annotations/train_coco.json",
        split="train",
        transforms=train_transform)
train_loader=DataLoader(ds_train,
                        batch_size=8,
                        collate_fn=collate_fn,
                        num_workers=2,
                        shuffle=True,
                        pin_memory=True)

ds_val = VisDroneData(
        json_path="dataset/visdrone/annotations/val_coco.json",
        split="val",
        transforms=train_transform)
val_loader=DataLoader(ds_val,
                      batch_size=8,
                      collate_fn=collate_fn,
                      num_workers=2,
                      shuffle=False,
                      pin_memory=True)

# take a batch
batch=next(iter(train_loader))
print(batch)

{'pixel_values': tensor([[[[0.1608, 0.1412, 0.1373,  ..., 0.3412, 0.3333, 0.3451],
          [0.1569, 0.1412, 0.1333,  ..., 0.3647, 0.3765, 0.3725],
          [0.1490, 0.1333, 0.1255,  ..., 0.3765, 0.3765, 0.3529],
          ...,
          [0.1098, 0.1020, 0.1059,  ..., 0.0471, 0.0431, 0.0549],
          [0.1098, 0.1059, 0.1098,  ..., 0.0549, 0.0510, 0.0627],
          [0.1098, 0.1059, 0.1137,  ..., 0.0627, 0.0549, 0.0667]],

         [[0.1647, 0.1451, 0.1412,  ..., 0.4118, 0.4039, 0.4157],
          [0.1608, 0.1451, 0.1373,  ..., 0.4353, 0.4471, 0.4431],
          [0.1529, 0.1373, 0.1294,  ..., 0.4471, 0.4471, 0.4235],
          ...,
          [0.1020, 0.0941, 0.0980,  ..., 0.0549, 0.0510, 0.0627],
          [0.1020, 0.0980, 0.1020,  ..., 0.0627, 0.0588, 0.0706],
          [0.1020, 0.0980, 0.1059,  ..., 0.0706, 0.0627, 0.0745]],

         [[0.1725, 0.1529, 0.1490,  ..., 0.4510, 0.4431, 0.4549],
          [0.1686, 0.1529, 0.1451,  ..., 0.4745, 0.4863, 0.4824],
          [0.1608, 0.1451

In [ ]:
import torch
import numpy as np
import supervision as sv

from dataclasses import dataclass, replace
from transformers import (
    AutoImageProcessor,
    TrainingArguments,
    Trainer
)
from torchmetrics.detection.mean_ap import MeanAveragePrecision

## 3. Train

In [ ]:
from torch.nn.utils import clip_grad_norm_
from transformers import get_scheduler
from tqdm import tqdm
import supervision as sv

def train(model, loader, optimizer, criterion, scheduler=None, device="cpu", max_grad_norm=0.1):
    model.train()
    loss_total = 0
    num_batches = 0

    # Initialize tqdm progress bar
    progress_bar = tqdm(loader, desc="Training", leave=True)

    for batch in progress_bar:
        # Move data to the specified device
        batch_images = batch['pixel_values'].to(device)
        batch_targets = [{k: v.to(device) for k, v in t.items()} for t in batch['labels']]

        # Forward pass
        optimizer.zero_grad()
        batch_outputs = model(batch_images, batch_targets)
        loss_dict = criterion(batch_outputs, batch_targets)
        loss = sum(loss_dict.values())
        loss_total += loss.item()
        num_batches += 1

        # Backward pass
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        # Optimizer step
        optimizer.step()

        # Scheduler step (if available)
        if scheduler:
            scheduler.step()

        # Update tqdm bar with the current loss
        progress_bar.set_postfix({"Batch Loss": loss.item()})

    # Close tqdm bar
    progress_bar.close()

    # Return average loss
    return loss_total / num_batches



# model output as a class -> to suit post processing method
@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


# compute mAP50 and mAP50-100 in validation
def validate(model, loader, processor, threshold, device):
    model.eval()

    # Initialize tqdm progress bar and evaluator
    progress_bar = tqdm(loader, desc="Validating", leave=True)
    evaluator = MeanAveragePrecision(box_format="xyxy", class_metrics=True)
    evaluator.warn_on_many_detections = False

    for batch in progress_bar:
        # Move batch data to the correct device
        images = batch['pixel_values'].to(device)
        batch_targets = batch['labels']

        # (1) Prepare target sizes and targets
        target_sizes = torch.tensor(np.array([x["orig_size"] for x in batch_targets])).to(device)
        batch_targets_processed = []

        # loop through individual targets
        for target, (height,width) in zip(batch_targets,target_sizes):
            boxes=target['boxes'].numpy()
            # convert to xyxy and compute actual dimensions
            boxes=sv.xcycwh_to_xyxy(boxes)
            boxes=boxes*np.array([width,height,width,height])
            boxes=torch.tensor(boxes, device=device)
            labels=target["labels"].to(device)
            batch_targets_processed.append({
                "boxes": boxes,
                "labels": labels
            })

        # (2) Compute predictions and post-process them
        with torch.no_grad():
            preds = model(images)
            outputs = ModelOutput(
                logits=preds['pred_logits'],
                pred_boxes=preds['pred_boxes']
            )
            batch_preds_processed = processor.post_process_object_detection(
                outputs,
                threshold=threshold,
                target_sizes=target_sizes
            )

        # (3) Update evaluator incrementally
        preds_for_evaluator = [
            {
                "boxes": pred["boxes"].cpu(),
                "scores": pred["scores"].cpu(),
                "labels": pred["labels"].cpu()
            }
            for pred in batch_preds_processed
        ]
        targets_for_evaluator = [
            {
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu()
            }
            for target in batch_targets_processed
        ]
        evaluator.update(preds=preds_for_evaluator, target=targets_for_evaluator)

    # Compute final metrics
    metrics = evaluator.compute()
    mAP50 = metrics["map_50"].item()
    mAP50_95 = metrics["map"].item()

    print(f"mAP@50: {mAP50:.4f}, mAP@50-95: {mAP50_95:.4f}")
    return mAP50, mAP50_95

In [ ]:
num_epochs=2
threshold=0.01

# Optimizer
learning_rate = 4e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Scheduler (linear warmup)
num_training_steps = len(train_loader) * num_epochs  # Total number of steps
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=10,
    num_training_steps=num_training_steps)

# post processor
# processor for evaluator
processor=AutoImageProcessor.from_pretrained(
            "PekingU/rtdetr_r18vd_coco_o365",
            do_resize=True,
            size={"width": 640, "height": 640},)

best_model=None
best_map50=0
device="cuda"
model.to(device)

for epoch in range(num_epochs):
    loss = train(model=model,loader=val_loader,optimizer=optimizer,
                 criterion=criterion,scheduler=scheduler,device=device,max_grad_norm=0.1)
    map50,map50_95=validate(model, val_loader, processor=processor,threshold=threshold)
    print(f"Epoch {epoch + 1}/{num_epochs} \nLoss: {loss:.4f} | map50: {map50} | map50_95: {map50_95}")
    if map50>best_map50:
        best_map50=map50
        best_model=model

Validating: 100%|██████████| 69/69 [00:32<00:00,  2.10it/s]


mAP@50: 0.0000, mAP@50-95: 0.0000
Epoch 1/2 | Loss: 19.6317 | map50: 7.663009455427527e-06 | map50_95: 1.3872096360501018e-06


Validating: 100%|██████████| 69/69 [00:39<00:00,  1.73it/s]


mAP@50: 0.0000, mAP@50-95: 0.0000
Epoch 2/2 | Loss: 19.0912 | map50: 1.550893830426503e-05 | map50_95: 2.412875346635701e-06
